# Phase 2 evidence export

Run this in Databricks after the isolated drift pipeline. It performs read-only validation and writes a Jupyter notebook with the actual query results to the demo Volume. The generated file can be downloaded and committed under docs/evidence/.


In [ ]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

dbutils.widgets.text("demo_batch_id", "")
demo_batch_id = dbutils.widgets.get("demo_batch_id").strip()
if not re.fullmatch(r"demo_[0-9a-f]{24}", demo_batch_id):
    raise ValueError("Set demo_batch_id to the batch printed by 98_prepare_drift_demo.ipynb")

def query_rows(sql):
    return [row.asDict(recursive=True) for row in spark.sql(sql).collect()]

queries = [
    ("Bronze table schema", "DESCRIBE TABLE carewatch.pipeline.bronze_nh_provider_info"),
    ("Silver table schema", "DESCRIBE TABLE carewatch.pipeline.silver_facility"),
    ("Bronze load timestamps", """SELECT 'bronze_nh_health_deficiencies' AS table_name, count(*) AS rows, count_if(load_timestamp IS NULL) AS missing_timestamps FROM carewatch.pipeline.bronze_nh_health_deficiencies
UNION ALL SELECT 'bronze_nh_penalties', count(*), count_if(load_timestamp IS NULL) FROM carewatch.pipeline.bronze_nh_penalties
UNION ALL SELECT 'bronze_nh_provider_info', count(*), count_if(load_timestamp IS NULL) FROM carewatch.pipeline.bronze_nh_provider_info
UNION ALL SELECT 'bronze_nh_mds_quality', count(*), count_if(load_timestamp IS NULL) FROM carewatch.pipeline.bronze_nh_mds_quality"""),
    ("Silver keys and load timestamps", """SELECT 'silver_deficiency' AS table_name, count(*) AS rows, count(DISTINCT deficiency_key) AS distinct_keys, count_if(load_timestamp IS NULL) AS missing_timestamps FROM carewatch.pipeline.silver_deficiency
UNION ALL SELECT 'silver_penalty', count(*), count(DISTINCT penalty_key), count_if(load_timestamp IS NULL) FROM carewatch.pipeline.silver_penalty
UNION ALL SELECT 'silver_facility', count(*), count(DISTINCT facility_key), count_if(load_timestamp IS NULL) FROM carewatch.pipeline.silver_facility
UNION ALL SELECT 'silver_mds_quality', count(*), count(DISTINCT mds_key), count_if(load_timestamp IS NULL) FROM carewatch.pipeline.silver_mds_quality"""),
    ("Historical parameterized MERGE", """SELECT pipeline_layer, dataset, load_type, parameter_processed, status, rows_read, rows_inserted, rows_updated, start_time, end_time
FROM carewatch.pipeline.pipeline_execution_logs
WHERE batch_id = '14c281a6e8c6c099c0dbad46'
ORDER BY start_time"""),
    ("Schema drift event", f"""SELECT drift_type, column_name, detail, load_timestamp
FROM carewatch.phase2_demo.schema_drift_log
WHERE batch_id = '{demo_batch_id}'"""),
    ("Drift Bronze rows", f"""SELECT count(*) AS bronze_rows, count_if(phase2_demo_added_column IS NOT NULL) AS rows_with_added_column
FROM carewatch.phase2_demo.bronze_nh_provider_info
WHERE _batch_id = '{demo_batch_id}'"""),
    ("Drift Silver and quarantine", f"""SELECT
  (SELECT count(*) FROM carewatch.phase2_demo.silver_facility WHERE source_batch_id = '{demo_batch_id}') AS accepted_silver_rows,
  (SELECT count(*) FROM carewatch.phase2_demo.silver_quarantine WHERE source_batch_id = '{demo_batch_id}') AS quarantined_rows,
  (SELECT count(*) FROM carewatch.phase2_demo.silver_quarantine WHERE source_batch_id = '{demo_batch_id}' AND array_join(failed_rules, ',') LIKE '%SAFE_CAST%') AS safe_cast_quarantine_rows"""),
    ("Full and incremental logs", """SELECT pipeline_layer, load_type, status, count(*) AS runs
FROM carewatch.pipeline.pipeline_execution_logs
GROUP BY pipeline_layer, load_type, status
ORDER BY pipeline_layer, load_type, status"""),
]

results = [(title, sql, query_rows(sql)) for title, sql in queries]
by_title = {title: rows for title, _, rows in results}
assert all(row["missing_timestamps"] == 0 for row in by_title["Bronze load timestamps"])
assert all(row["missing_timestamps"] == 0 and row["rows"] == row["distinct_keys"] for row in by_title["Silver keys and load timestamps"])
history = [row for row in by_title["Historical parameterized MERGE"] if row["pipeline_layer"] == "Bronze-to-Silver" and row["status"] == "SUCCESS"]
assert any(row["rows_inserted"] > 0 for row in history)
assert any("reprocess=true" in row["parameter_processed"] and row["rows_inserted"] == 0 and row["rows_updated"] == 0 for row in history)
assert any(row["drift_type"] == "ADDED_COLUMN" and row["column_name"] == "phase2_demo_added_column" for row in by_title["Schema drift event"])
assert by_title["Drift Bronze rows"][0] == {"bronze_rows": 2, "rows_with_added_column": 2}
assert by_title["Drift Silver and quarantine"][0] == {"accepted_silver_rows": 1, "quarantined_rows": 1, "safe_cast_quarantine_rows": 1}
coverage = {(row["pipeline_layer"], row["load_type"]) for row in by_title["Full and incremental logs"] if row["status"] in ("SUCCESS", "QUARANTINED_PARTIAL")}
assert all((layer, kind) in coverage for layer in ("Raw-to-Bronze", "Bronze-to-Silver") for kind in ("full", "incremental"))

def markdown(value):
    return {"cell_type": "markdown", "metadata": {}, "source": [line + "\n" for line in value.split("\n")]}

def code_with_output(sql, rows, index):
    rendered = json.dumps(rows, indent=2, default=str, ensure_ascii=False)
    return {
        "cell_type": "code",
        "execution_count": index,
        "metadata": {"execution_provenance": "Captured by Databricks Spark job"},
        "source": ["[row.asDict(recursive=True) for row in spark.sql(" + repr(sql) + " ).collect()]\n"],
        "outputs": [{
            "output_type": "execute_result",
            "execution_count": index,
            "metadata": {},
            "data": {"text/plain": [rendered]}
        }]
    }

stamp = datetime.now(timezone.utc).isoformat()
cells = [markdown("# CareWatch Phase 2 live notebook outputs\nGenerated inside Databricks at " + stamp + " using the linked pipeline tables. These are actual Spark SQL query results from the completed jobs; this file was generated by 99_export_phase2_evidence.ipynb.")]
for index, (title, sql, rows) in enumerate(results, start=1):
    cells.append(markdown("## " + title))
    cells.append(code_with_output(sql, rows, index))
cells.append(markdown("## Verification result"))
cells.append({
    "cell_type": "code",
    "execution_count": len(results) + 1,
    "metadata": {},
    "source": ["print('All Phase 2 evidence checks passed.')\n"],
    "outputs": [{"output_type": "stream", "name": "stdout", "text": ["All Phase 2 evidence checks passed.\n"]}]
})
exported = {
    "cells": cells,
    "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}, "language_info": {"name": "python"}},
    "nbformat": 4,
    "nbformat_minor": 0
}
destination = Path("/Volumes/carewatch/phase2_demo/landing/evidence/phase2_live_outputs.ipynb")
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(exported, indent=2, default=str, ensure_ascii=False), encoding="utf-8")
print("All Phase 2 evidence checks passed.")
print("Saved actual Spark query outputs to", destination)
dbutils.notebook.exit(str(destination))
